<a href="https://colab.research.google.com/github/HY-BME/bme_test_files/blob/%EB%8D%B0%EC%9D%B4%ED%84%B0%EA%B8%B0%EB%B0%98%ED%97%AC%EC%8A%A4%EC%BC%80%EC%96%B4%EA%B8%B0%EC%88%A0/class_0317.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import pandas as pd

# 데이터 로드
df = pd.read_csv('/content/sample_data/raw_medical_data.csv')

print("정제 전 컬럼명:", df.columns.tolist())
# 결과: [' Patient_ID ', 'Age', ' hr ', 'OX ']

정제 전 컬럼명: [' Patient_ID ', 'Age', ' hr ', 'OX ']


In [23]:
df

#hr : 심박수
#OX : 산소포화농도

,Patient_ID,Age,hr,OX
0,P-001,65,85.9,96.2
1,P-002,22,57.9,96.7
2,P-003,48,73.3,95.3
3,P-004,54,64.7,93.8
4,P-005,58,NaN,99.5
5,P-006,37,41.4,95.6
6,P-007,39,53.7,100.3
7,P-008,62,66.6,98.6
8,P-009,79,86.1,96.4
9,P-010,77,72.9,94.8


In [24]:
#공백제거 및 소문자 변환
df.columns = df.columns.str.strip().str.lower();
#컬럼명 일괄 변경
df.rename(columns={'hr': 'heart_rate', 'ox' : 'spo2'}, inplace=True);

df

,patient_id,age,heart_rate,spo2
0,P-001,65,85.9,96.2
1,P-002,22,57.9,96.7
2,P-003,48,73.3,95.3
3,P-004,54,64.7,93.8
4,P-005,58,NaN,99.5
5,P-006,37,41.4,95.6
6,P-007,39,53.7,100.3
7,P-008,62,66.6,98.6
8,P-009,79,86.1,96.4
9,P-010,77,72.9,94.8


In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   patient_id  50 non-null     object 
 1   age         50 non-null     int64  
 2   heart_rate  47 non-null     float64
 3   spo2        50 non-null     float64
dtypes: float64(2), int64(1), object(1)
memory usage: 1.7+ KB


In [26]:
df['spo2'] = pd.to_numeric(df['spo2'], errors='coerce') #object type을 float로 치환
df.info()

df

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   patient_id  50 non-null     object 
 1   age         50 non-null     int64  
 2   heart_rate  47 non-null     float64
 3   spo2        50 non-null     float64
dtypes: float64(2), int64(1), object(1)
memory usage: 1.7+ KB


,patient_id,age,heart_rate,spo2
0,P-001,65,85.9,96.2
1,P-002,22,57.9,96.7
2,P-003,48,73.3,95.3
3,P-004,54,64.7,93.8
4,P-005,58,NaN,99.5
5,P-006,37,41.4,95.6
6,P-007,39,53.7,100.3
7,P-008,62,66.6,98.6
8,P-009,79,86.1,96.4
9,P-010,77,72.9,94.8


In [27]:
#결측값 처리
print(df.isnull())

#결측값 개수 확인
print(df.isnull().sum())

patient_id    0
age           0
heart_rate    3
spo2          0
dtype: int64


In [35]:
#결측치 행 삭제
#df_cleaned = df.dropna(subset=['heart_rate'])

#결측값 대체
#심박수 평균값으로 결측치 채우기
df['heart_rate'] = df['heart_rate'].fillna(df['heart_rate'].mean())

# 시계열 데이터의 경우, 이전 값으로 채우기 (Forward Fill)
#df['spo2'] = df['spo2'].fillna(method='ffill')

df.isnull().sum()

,0
patient_id,0
age,0
heart_rate,0
spo2,0


In [36]:
#이상치 처리 - 생리학적으로 불가능한 수치나 기계적 오류

# Q1 = df['heart_rate'].quantile(0.25)
# Q3 = df['heart_rate'].quantile(0.75)
# IQR = Q3 - Q1

# # IQR 범위를 벗어나는 데이터 필터링
# lower_bound = Q1 - 1.5 * IQR #1.5배 만큼의 거리 최소
# upper_bound = Q3 + 1.5 * IQR #1.5배 만큼의 거리 최대

# df_final = df[(df['heart_rate'] >= lower_bound) & (df['heart_rate'] <= upper_bound)]

#spo2 기준
Q1 = df['spo2'].quantile(0.25)
Q3 = df['spo2'].quantile(0.75)
IQR = Q3 - Q1

# IQR 범위를 벗어나는 데이터 필터링
lower_bound = Q1 - 1.5 * IQR #1.5배 만큼의 거리 최소
upper_bound = Q3 + 1.5 * IQR #1.5배 만큼의 거리 최대

df_final = df[(df['spo2'] >= lower_bound) & (df['spo2'] <= upper_bound)]

df.describe()

,age,heart_rate,spo2
count,50.000000,50.000000,50.000000
mean,51.540000,75.338298,96.962000
std,16.152601,13.704454,2.224987
min,22.000000,41.400000,92.900000
25%,40.000000,65.075000,95.500000
50%,52.500000,75.369149,96.850000
75%,65.750000,85.850000,98.475000
max,79.000000,103.700000,102.200000


In [40]:
df_final = df_final[(df_final['spo2'] >= 0) & (df_final['spo2'] <= 100)] #산소포화도 0이상 100이하
df_final

,patient_id,age,heart_rate,spo2
0,P-001,65,85.900000,96.2
1,P-002,22,57.900000,96.7
2,P-003,48,73.300000,95.3
3,P-004,54,64.700000,93.8
4,P-005,58,75.338298,99.5
5,P-006,37,41.400000,95.6
7,P-008,62,66.600000,98.6
8,P-009,79,86.100000,96.4
9,P-010,77,72.900000,94.8
10,P-011,42,75.000000,95.5
